# LINet Training on SUN RGB-D - Google Colab

**Hyperparameter tuning with Ray Tune using a locked 80/20 train/val split**

---

## Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime > Change runtime type > A100
- [ ] **Upload dataset to Drive:** `MyDrive/datasets/sunrgbd_19_hha.tar.gz`


## 1. Environment Setup & GPU Verification

In [17]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

# Check PyTorch and CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    # Check if it's A100
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\n✅ A100 GPU detected - PERFECT for training!")
    elif 'V100' in gpu_name:
        print("\n✅ V100 GPU detected - Good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\n⚠️  T4 GPU detected - Will be slower, consider upgrading to A100")
    else:
        print(f"\n⚠️  GPU: {gpu_name} - Consider using A100 for best performance")
else:
    print("\n❌ NO GPU DETECTED!")
    print("Please enable GPU: Runtime → Change runtime type → Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

GPU VERIFICATION
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU Device: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU Memory: 94.97 GB

⚠️  GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition - Consider using A100 for best performance



In [18]:
# Detailed GPU info
!nvidia-smi

Thu Apr 23 07:01:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             48W /  600W |      68MiB /  97887MiB |      0%   E. Process |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [19]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Google Drive mounted successfully!

Drive contents:
total 3118192
-rw------- 1 root root        176 Sep 21  2019 06-lab2.gdoc
-rw------- 1 root root      21621 Sep 30  2024 113-1363667-3121001@USSR24093000064918@pre-paid.png
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (1).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (2).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (3).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final.gdoc
-rw------- 1 root root        176 Jul 11  2025 2025_Gabriel_Clinger_Contractor Agreement_BASE copy.gdoc
-rw------- 1 root root      32204 Apr 18  2022 2900 On First- Welcome Home Next Steps.docx
-rw------- 1 root root       8822 Jun 24  2017 A6.docx
-rw------- 1 root root      22204 Jan 21  2023 activity (1).xlsx
-rw------- 1 root root      22161 Ja

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [20]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"  # UPDATE THIS
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"  # Local copy for fast I/O

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

# Ensure we're in a valid directory
os.chdir('/content')
print(f"Starting in: {os.getcwd()}")

# Check if repo already exists (same session, rerunning cell)
if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"\n📁 Repo already exists: {LOCAL_REPO_PATH}")
    print(f"🔄 Pulling latest changes...")

    os.chdir(LOCAL_REPO_PATH)
    !git pull
    print("✅ Repo updated")

# Clone from GitHub (first run)
else:
    # Remove old incomplete copy if exists
    if Path(LOCAL_REPO_PATH).exists():
        print(f"\n🗑️  Removing incomplete repo copy...")
        !rm -rf {LOCAL_REPO_PATH}

    print(f"\n🔄 Cloning from GitHub...")
    print(f"   Repo: {GITHUB_REPO}")
    print(f"   Destination: {LOCAL_REPO_PATH}")

    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}

    # Verify clone succeeded
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository to {LOCAL_REPO_PATH}")

    print("✅ Repo cloned successfully")
    os.chdir(LOCAL_REPO_PATH)

# Verify repo structure
print(f"\n📂 Repository structure:")
!ls -la {LOCAL_REPO_PATH}

print(f"\n✅ Working directory: {os.getcwd()}")

REPOSITORY SETUP
Starting in: /content

📁 Repo already exists: /content/Multi-Stream-Neural-Networks
🔄 Pulling latest changes...
Already up to date.
✅ Repo updated

📂 Repository structure:
total 88
drwxr-xr-x 12 root root  4096 Apr 22 23:51 .
drwxr-xr-x  1 root root  4096 Apr 22 23:52 ..
drwxr-xr-x  5 root root  4096 Apr 22 23:51 configs
drwxr-xr-x  2 root root  4096 Apr 22 23:51 data
drwxr-xr-x  5 root root  4096 Apr 22 23:51 docs
drwxr-xr-x  3 root root  4096 Apr 22 23:51 experiments
drwxr-xr-x  9 root root  4096 Apr 23 07:01 .git
-rw-r--r--  1 root root   732 Apr 22 23:51 .gitattributes
drwxr-xr-x  3 root root  4096 Apr 22 23:51 .github
-rw-r--r--  1 root root   847 Apr 22 23:51 .gitignore
-rw-r--r--  1 root root  1084 Apr 22 23:51 LICENSE
drwxr-xr-x  2 root root  4096 Apr 22 23:51 notebooks
-rw-r--r--  1 root root   198 Apr 22 23:51 pytest.ini
-rw-r--r--  1 root root  6920 Apr 22 23:51 README.md
-rw-r--r--  1 root root   126 Apr 22 23:51 requirements.txt
drwxr-xr-x  2 root root  40

## 4. Install Dependencies

In [21]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] optuna kornia

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import ray
import kornia

print("✅ All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   ray: {ray.__version__}")
print(f"   kornia: {kornia.__version__}")


Installing dependencies...
✅ All dependencies installed!
   h5py: 3.16.0
   matplotlib: 3.10.0
   ray: 2.55.1
   kornia: 0.8.2


## 5. Copy SUN RGB-D Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** SUN RGB-D 19-category preprocessed dataset with RGB + Depth


In [22]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_hha.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/sunrgbd_19_hha"

print("=" * 60)
print("SUN RGB-D 19-CATEGORY DATASET SETUP")
print("=" * 60)

if Path(LOCAL_DATASET_PATH).exists():
    print(f"Already on local disk: {LOCAL_DATASET_PATH}")
    train_count = len(list(Path(f"{LOCAL_DATASET_PATH}/train/rgb").glob("*.png")))
    print(f"   Train samples: {train_count}")
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found on Drive: {DRIVE_DATASET_TAR}")
    tar_name = Path(DRIVE_DATASET_TAR).name
    local_tar = f"/dev/shm/{tar_name}"
    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    train_count = len(list(Path(f"{LOCAL_DATASET_PATH}/train/rgb").glob("*.png")))
    print(f"Extracted. Train samples: {train_count}")
else:
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET_TAR}")

print(f"\nDataset ready at: {LOCAL_DATASET_PATH}")


SUN RGB-D 19-CATEGORY DATASET SETUP
Already on local disk: /dev/shm/sunrgbd_19_traintest
   Train samples: 0

Dataset ready at: /dev/shm/sunrgbd_19_traintest


## 6. Setup Python Path & Import LINet


In [23]:
import sys
import os

modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project structure:")
!ls -la {project_root}/src/models/

print("\nImporting LiNet and dataloaders...")
from src.models.linear_integration.li_net3 import li_resnet18
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders, SUNRGBDDataset
from src.training.augmentation_config import AugmentationConfig

from ray import train, tune
from ray.tune.schedulers import ASHAScheduler

print("All imports successful!")


Project structure:
total 52
drwxr-xr-x 12 root root 4096 Apr 22 23:52 .
drwxr-xr-x  8 root root 4096 Apr 22 23:52 ..
drwxr-xr-x  3 root root 4096 Apr 22 23:52 abstracts
drwxr-xr-x  3 root root 4096 Apr 22 23:52 common
drwxr-xr-x  3 root root 4096 Apr 22 23:52 core
drwxr-xr-x  2 root root 4096 Apr 22 23:51 direct_mixing_activation
drwxr-xr-x  2 root root 4096 Apr 22 23:51 direct_mixing_bn
drwxr-xr-x  2 root root 4096 Apr 22 23:51 direct_mixing_conv
-rw-r--r--  1 root root 1076 Apr 22 23:51 __init__.py
drwxr-xr-x  5 root root 4096 Apr 22 23:52 linear_integration
drwxr-xr-x  3 root root 4096 Apr 22 23:52 multi_channel
drwxr-xr-x  2 root root 4096 Apr 22 23:52 __pycache__
drwxr-xr-x  2 root root 4096 Apr 22 23:51 utils

Importing LiNet and dataloaders...
All imports successful!


## 8b. Hyperparameter Tuning with Ray Tune

- **Parallel Trials:** Run multiple configurations simultaneously
- **Locked 80/20 Split:** Deterministic stratified train/val split
- **ASHA Scheduler:** Early-stop unpromising trials
- **Resumable:** Experiment state saved to Google Drive, survives Colab restarts
- **Optional Pretrained Weights:** Load Omni backbone for transfer learning


In [24]:
import os
import time

# 1. Define Paths explicitly
mps_pipe_dir = "/tmp/nvidia-mps"
mps_log_dir = "/tmp/nvidia-log"

# 2. Create the directories (CRITICAL: Daemon fails if log dir doesn't exist)
os.makedirs(mps_pipe_dir, exist_ok=True)
os.makedirs(mps_log_dir, exist_ok=True)

# 3. Set Environment Variables for the current Python process
os.environ["CUDA_MPS_PIPE_DIRECTORY"] = mps_pipe_dir
os.environ["CUDA_MPS_LOG_DIRECTORY"] = mps_log_dir
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# 4. Configure GPU and Start Daemon using the SAME environment variables
# We use f-strings to pass the python variables into the shell command
print("Setting GPU to Exclusive Process Mode...")
!nvidia-smi -i 0 -c EXCLUSIVE_PROCESS

print("Starting MPS Daemon...")
# We explicitly pass the env vars to the shell command
!export CUDA_MPS_PIPE_DIRECTORY={mps_pipe_dir} && \
 export CUDA_MPS_LOG_DIRECTORY={mps_log_dir} && \
 nvidia-cuda-mps-control -d

# 5. Verify it is running
print("Verifying Daemon Status...")
time.sleep(1) # Give it a second to start
!ps -ef | grep mps

# Check if the pipe file actually exists
if os.path.exists(os.path.join(mps_pipe_dir, "control")):
    print("✅ MPS Control Pipe found. Setup success.")
else:
    print("❌ MPS Control Pipe NOT found. Check /tmp/nvidia-log for errors.")
    # Optional: Print logs if it failed
    !cat {mps_log_dir}/control.log

Setting GPU to Exclusive Process Mode...
Compute mode is already set to EXCLUSIVE_PROCESS for GPU 00000000:05:00.0.
All done.
Starting MPS Daemon...
An instance of this daemon is already running
Verifying Daemon Status...
root        6142       1  0 Apr22 ?        00:00:00 nvidia-cuda-mps-control -d
root        9317    6142  0 Apr22 ?        00:00:06 nvidia-cuda-mps-server
root      197054    4748  0 07:01 ?        00:00:00 /bin/bash -c ps -ef | grep mps
root      197056  197054  0 07:01 ?        00:00:00 grep mps
✅ MPS Control Pipe found. Setup success.


In [ ]:
import random
import numpy as np

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch
from optuna.samplers import TPESampler
from ray.tune.schedulers import MedianStoppingRule
import torch
from collections import Counter

from src.models.linear_integration.li_net3 import li_resnet18
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.data_utils.sunrgbd_dataset import SUNRGBDDataset
from src.training.augmentation_config import AugmentationConfig
from src.utils.seed import set_seed
from src.data_utils.sunrgbd_dataset import _load_norm_stats
from src.models.common.model_helpers import load_pretrained_backbone


class TrialTerminated(Exception):
    """Raised when a trial should be terminated early."""
    pass


class RayTuneReporter:
    """Callback for reporting metrics to Ray Tune during training."""

    SMOOTHING_WINDOW = 5  # epochs to average val_mca over

    def __init__(self):
        self.best_accuracy = 0.0
        self.best_loss = float('inf')
        self.best_train_acc = 0.0
        self.best_val_mca = 0.0
        self.best_train_mca = 0.0
        self.best_epoch = 0
        # Smoothed val_mca tracking
        self.val_mca_history = []
        self.best_smoothed_val_mca = 0.0

    def on_epoch_end(self, epoch, logs):
        """Report current AND best metrics to Ray Tune."""
        if logs['val_accuracy'] > self.best_accuracy:
            self.best_accuracy = logs['val_accuracy']
            if logs['train_accuracy'] > self.best_train_acc:
                self.best_train_acc = logs['train_accuracy']

        if logs['val_loss'] < self.best_loss:
            self.best_loss = logs['val_loss']

        val_mca = logs.get('val_mca', 0.0)
        train_mca = logs.get('train_mca', 0.0)
        if val_mca > self.best_val_mca:
            self.best_val_mca = val_mca
            self.best_epoch = epoch
            if train_mca > self.best_train_mca:
                self.best_train_mca = train_mca

        # Smoothed val_mca: rolling average over last N epochs
        self.val_mca_history.append(val_mca)
        if len(self.val_mca_history) > self.SMOOTHING_WINDOW:
            self.val_mca_history.pop(0)
        window = len(self.val_mca_history)  # always min(epoch+1, SMOOTHING_WINDOW)
        smoothed_val_mca = sum(self.val_mca_history) / window
        if smoothed_val_mca > self.best_smoothed_val_mca:
            self.best_smoothed_val_mca = smoothed_val_mca

        # Legacy composite (kept for diagnostic/comparison; HPO should use smoothed_val_mca)
        gap = self.best_train_mca - self.best_val_mca
        composite = self.best_val_mca - 2 * (gap**3)

        tune.report({
            "accuracy": logs['val_accuracy'],
            "loss": logs['val_loss'],
            "best_accuracy": self.best_accuracy,
            "best_loss": self.best_loss,
            "train_loss": logs['train_loss'],
            "train_accuracy": logs['train_accuracy'],
            "best_train_acc": self.best_train_acc,
            "val_mca": val_mca,
            "train_mca": train_mca,
            "best_val_mca": self.best_val_mca,
            "best_train_mca": self.best_train_mca,
            "smoothed_val_mca": smoothed_val_mca,
            "best_smoothed_val_mca": self.best_smoothed_val_mca,
            "composite": composite,
            "gap": gap,
            "best_epoch": self.best_epoch,
        })


def train_linet_tune(
    config,
    data_root=None,
    norm_stats=None,
    pretrained_weights_path=None,
    freeze_backbone_epochs=0,
    freeze_backbone_lr=1e-3,
    seed=42,
    train_indices=None,
    val_indices=None,
):
    """
    Trainable function for Ray Tune — locked 80/20 stratified split.

    Args:
        config: Ray Tune configuration dict with hyperparameters
        data_root: Path to dataset root (with train/ directory)
        norm_stats: Normalization statistics dict
        pretrained_weights_path: Path to pretrained checkpoint (or None)
        seed: Random seed for reproducible trials
    """
    set_seed(seed, deterministic=False)
    g = torch.Generator().manual_seed(seed)

    # Per-trial augmentation config
    aug_config = AugmentationConfig(
        rgb_aug_prob=config.get("rgb_aug_prob"),
        rgb_aug_mag=config.get("rgb_aug_mag"),
        depth_aug_prob=config.get("depth_aug_prob"),
        depth_aug_mag=config.get("depth_aug_mag"),
    )

    # Two dataset instances from the same train/ directory:
    # 1) train_dataset: augmentation ON (split='train')
    # 2) val_dataset:   augmentation OFF (split overridden to 'val')
    train_dataset = SUNRGBDDataset(
        data_root=data_root,
        split='train',
        normalize=True,  # CPU pipeline normalizes inside the dataset (HHA)
        **aug_config.to_dict(),
     use_hha=True,
)
    val_dataset = SUNRGBDDataset(
        data_root=data_root,
        split='train',
        normalize=True,
     use_hha=True,
)
    val_dataset.split = 'val'  # Disable augmentation in __getitem__

    # Locked 80/20 stratified split (deterministic — same split every trial)
    all_labels = train_dataset.labels

    train_subset = torch.utils.data.Subset(train_dataset, train_indices)
    val_subset = torch.utils.data.Subset(val_dataset, val_indices)

    # Stratified sampling for training
    subset_labels = [all_labels[i] for i in train_indices]
    label_counts = Counter(subset_labels)
    num_samples = len(subset_labels)
    class_weights = {label: num_samples / count for label, count in label_counts.items()}
    sample_weights = torch.tensor(
        [class_weights[label] for label in subset_labels], dtype=torch.float32
    )

    train_sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=num_samples,
        replacement=True,
        generator=g,
    )

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    train_loader = torch.utils.data.DataLoader(
        train_subset,
        batch_size=64,
        shuffle=False,
        sampler=train_sampler,
        num_workers=2,
        prefetch_factor=2,
        persistent_workers=True,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )
    val_loader = torch.utils.data.DataLoader(
        val_subset,
        batch_size=64,
        shuffle=False,
        num_workers=1,
        prefetch_factor=2,
        persistent_workers=False,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )

    # Create Model
    model = li_resnet18(
        num_classes=19,
        stream_input_channels=[3, 3],
        dropout_p=config["dropout_p"],
        width_multiplier=0.75,
        device="cuda",
        use_amp=True,
    )

    # Load pretrained backbone weights (if provided)
    if pretrained_weights_path is not None:
        load_pretrained_backbone(model, pretrained_weights_path, verbose=False)

        # Optional backbone freeze warmup: train only classifier head with static LR
        if freeze_backbone_epochs > 0:
            for param in model.parameters():
                param.requires_grad = False
            for param in model.fc.parameters():
                param.requires_grad = True

            fc_optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=freeze_backbone_lr,
            )
            model.compile(
                optimizer=fc_optimizer,
                scheduler=None,
                loss='cross_entropy',
                label_smoothing=config["label_smoothing"],
                gpu_augmentation=False,
                # norm_stats=norm_stats,
                **aug_config.to_dict(),
            )
            model.fit(
                train_loader=train_loader,
                val_loader=val_loader,
                epochs=freeze_backbone_epochs,
                verbose=False,
                modality_dropout=False,
            )

            # Unfreeze all parameters for the full trial
            for param in model.parameters():
                param.requires_grad = True

    # Create Optimizer
    optimizer = create_stream_optimizer(
        model,
        optimizer_type='adamw',
        stream_lrs=[config["lr"], config["lr"]],
        stream_weight_decays=[config["wd"], config["wd"]],
        shared_lr=config["lr"],
        integration_weight_decay=config["wd"],
        stem_lr_multiplier=config["stem_lr_multiplier"],
    )

    # Create Scheduler
    if config['stem_lr_multiplier'] == 1.0:
        eta_min_list = [config['eta_min']] * 4  # 4 groups: stream0, stream1, integ, other
    else:
        eta_min_list = [
            config['eta_min'] * config['stem_lr_multiplier'],
            config['eta_min'] * config['stem_lr_multiplier'],
            config['eta_min'], config['eta_min'],
            config['eta_min'], config['eta_min'],
        ]
    warmup_epochs = 5
    scheduler = setup_scheduler(
        optimizer,
        scheduler_type='cosine',
        eta_min=eta_min_list,
        t_max=25,
        train_loader_len=len(train_loader),
        warmup_epochs=warmup_epochs,
        warmup_start_factor=0.2,
    )

    # Compile
    model.compile(
        optimizer=optimizer,
        scheduler=scheduler,
        loss='cross_entropy',
        label_smoothing=config["label_smoothing"],
        gpu_augmentation=False,
        # norm_stats=norm_stats,
        **aug_config.to_dict(),
    )

    # Train
    try:
        model.fit(
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=30,
            early_stopping=True,
            patience=15,
            monitor='val_mca',
            grad_clip_norm=config["grad_clip_norm"],
            modality_dropout=True,
            modality_dropout_start=0,
            modality_dropout_ramp=0,
            modality_dropout_rate=config['modality_dropout_rate'],
            callbacks=[RayTuneReporter()],
            verbose=False,
        )
    except TrialTerminated as e:
        print(f"\n{e}")


In [26]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# Ray Tune saves ALL experiment state to DRIVE_STORAGE_PATH via storage_path.
# When Colab dies, re-run the notebook — Tuner.restore() picks up where it
# left off. Completed trials preserved, interrupted trials restart.
# =============================================================================

import hashlib
import json as json_module
import os
import pandas as pd
from pathlib import Path

# --- Ray Tune persistent storage on Google Drive ---
DRIVE_STORAGE_PATH = "/content/drive/MyDrive/ray_tune_experiments"
LOCAL_STORAGE_PATH = "/content/ray_results"
EXPERIMENT_NAME = "scannet_sun_rgbd_hpo_MD"

SEED = 152
NUM_SAMPLES = 300  # Total trials to run across all sessions

# --- Pretrained Weights (Optional) ---
# Set LOAD_WEIGHTS = True to initialize every trial from pretrained backbone
# weights (e.g. from OmniObject3D pretraining). The fc head is skipped
# automatically if num_classes differs.
LOAD_WEIGHTS = False  # HHA: trains from random init
PRETRAINED_WEIGHTS_PATH = "/content/drive/MyDrive/linet_checkpoints/scannet_pretrain_20260414_211058_withMD/epoch_checkpoints/checkpoint_epoch_57.pt"

# --- Backbone Freeze Warmup (Optional, only used when LOAD_WEIGHTS = True) ---
# Freeze pretrained backbone for N epochs, training only the classifier head
# with a static LR before the full HPO trial begins.
FREEZE_BACKBONE_EPOCHS = 3    # Set > 0 to enable (e.g. 3-5 epochs)
FREEZE_BACKBONE_LR = 1e-3     # Static LR for classifier warmup


Path(DRIVE_STORAGE_PATH).mkdir(parents=True, exist_ok=True)
Path(LOCAL_STORAGE_PATH).mkdir(parents=True, exist_ok=True)

experiment_path = os.path.join(DRIVE_STORAGE_PATH, EXPERIMENT_NAME)
local_experiment_path = os.path.join(LOCAL_STORAGE_PATH, EXPERIMENT_NAME)
RESUME_EXISTING = os.path.exists(experiment_path)

if RESUME_EXISTING:
    # Validate experiment dir has actual content
    _exp_files = os.listdir(experiment_path) if os.path.isdir(experiment_path) else []
    if len(_exp_files) == 0:
        print(f"  WARNING: {experiment_path} exists but is empty \u2014 starting fresh")
        RESUME_EXISTING = False

print(f"Drive storage: {DRIVE_STORAGE_PATH}")
print(f"Local storage: {LOCAL_STORAGE_PATH}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Resume existing: {RESUME_EXISTING}")
print(f"Total trials: {NUM_SAMPLES}")
print(f"Load pretrained weights: {'ENABLED' if LOAD_WEIGHTS else 'DISABLED'}")
if LOAD_WEIGHTS:
    print(f"   Weights: {PRETRAINED_WEIGHTS_PATH}")
    print(f"   Freeze warmup: {FREEZE_BACKBONE_EPOCHS} epochs (LR={FREEZE_BACKBONE_LR})" if FREEZE_BACKBONE_EPOCHS > 0 else "   Freeze warmup: DISABLED")
if RESUME_EXISTING:
    print(f"\n  Previous experiment found at {experiment_path}")
    print(f"  Copying Drive -> local, then Tuner.restore() from local.")
    # Copy experiment state from Drive to local before Tuner.restore
    import subprocess as _sp_cfg
    os.makedirs(local_experiment_path, exist_ok=True)
    _result = _sp_cfg.run(
        ["rsync", "-a", experiment_path + "/", local_experiment_path + "/"],
        capture_output=True, text=True,
    )
    if _result.returncode == 0:
        print(f"  Restored to {local_experiment_path}")
    else:
        raise RuntimeError(f"Restore failed: {_result.stderr[:300]}")


Drive storage: /content/drive/MyDrive/ray_tune_experiments
Local storage: /content/ray_results
Experiment: scannet_sun_rgbd_hpo_MD
Resume existing: False
Total trials: 300
Load pretrained weights: ENABLED
   Weights: /content/drive/MyDrive/linet_checkpoints/scannet_pretrain_20260414_211058_withMD/epoch_checkpoints/checkpoint_epoch_57.pt
   Freeze warmup: 3 epochs (LR=0.001)


In [27]:
# Initialize Ray
import shutil
import subprocess
import time as _time

from ray.tune import CLIReporter
from ray.tune import Callback as TuneCallback

import json as _json_split
from sklearn.model_selection import StratifiedGroupKFold

os.environ["RAY_AIR_NEW_OUTPUT"] = "0"  # must be set BEFORE ray.init()

class DriveSyncCallback(TuneCallback):
    """Periodically rsyncs local Ray Tune experiment state to Google Drive.

    Ray Tune writes to LOCAL_STORAGE_PATH (fast local disk).  This callback
    rsyncs local -> Drive incrementally (only changed files) so that state
    survives Colab session death without blocking the Ray driver.
    """

    def __init__(self, local_storage_path, drive_storage_path, experiment_name,
                 sync_interval_seconds=300):
        self._local_path = os.path.join(local_storage_path, experiment_name)
        self._drive_path = os.path.join(drive_storage_path, experiment_name)
        self._sync_interval = sync_interval_seconds
        self._last_sync = 0.0

    def _sync(self, reason=""):
        if not os.path.isdir(self._local_path):
            return
        try:
            os.makedirs(self._drive_path, exist_ok=True)
            result = subprocess.run(
                ["rsync", "-a",
                 self._local_path + "/",
                 self._drive_path + "/"],
                capture_output=True, text=True, timeout=120,
            )
            if result.returncode == 0:
                self._last_sync = _time.time()
                print(f"[DriveSyncCallback] synced to Drive ({reason})")
            else:
                print(f"[DriveSyncCallback] WARNING: rsync failed: {result.stderr[:200]}")
        except subprocess.TimeoutExpired:
            print(f"[DriveSyncCallback] WARNING: rsync timed out (120s)")
        except Exception as e:
            print(f"[DriveSyncCallback] WARNING: sync failed: {e}")

    def on_trial_result(self, iteration, trials, trial, result, **info):
        if _time.time() - self._last_sync >= self._sync_interval:
            self._sync(reason=f"periodic, iter={result.get('training_iteration', '?')}")

    def on_trial_complete(self, iteration, trials, trial, **info):
        if _time.time() - self._last_sync >= 60:
            self._sync(reason="trial complete")

    def on_experiment_end(self, trials, **info):
        self._sync(reason="experiment end")


class BestTrialReporter(TuneCallback):
    """Periodically prints the best trial's config and metrics."""

    def __init__(self, metric="composite", mode="max", every_n_results=20):
        self._metric = metric
        self._mode = mode
        self._every_n = every_n_results
        self._result_count = 0
        self._best_value = float('-inf') if mode == "max" else float('inf')
        self._best_config = None
        self._best_epoch = None

    def on_trial_result(self, iteration, trials, trial, result, **info):
        self._result_count += 1
        val = result.get(self._metric, None)
        if val is None:
            return
        improved = (val > self._best_value) if self._mode == "max" else (val < self._best_value)
        if improved:
            self._best_value = val
            self._best_config = trial.config.copy()
            self._best_epoch = result.get("best_epoch", None)

        if self._result_count % self._every_n == 0 and self._best_config is not None:
            self._print_best(result)

    def _print_best(self, latest_result):
        print(f"\n{'─'*60}")
        print(f"  ★ Best {self._metric}: {self._best_value*100:.2f}% "
              f" best_epoch: {self._best_epoch}")
        # if self._best_epoch is not None:
        #     print(f"    best_epoch: {self._best_epoch}")
        for k, v in self._best_config.items():
            if isinstance(v, float):
                print(f"    {k}: {v:.2e}" if abs(v) < 0.01 else f"    {k}: {v:.4f}")
            else:
                print(f"    {k}: {v}")
        print(f"{'─'*60}")


ray.shutdown()
ray.init(
    ignore_reinit_error=True,
    runtime_env={
        "env_vars": {
            "CUDA_MPS_PIPE_DIRECTORY": "/tmp/nvidia-mps",
            "CUDA_MPS_LOG_DIRECTORY": "/tmp/nvidia-log",
            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
            "CUDA_VISIBLE_DEVICES": "0",
        }
    }
)

norm_stats = _load_norm_stats(LOCAL_DATASET_PATH)

# Pre-compute scene-aware split ONCE (not per-trial)
with open(os.path.join(LOCAL_DATASET_PATH, "train", "scene_groups.json")) as f:
    _scene_groups = _json_split.load(f)
with open(os.path.join(LOCAL_DATASET_PATH, "train", "labels.txt")) as f:
    _all_labels = [int(l) for l in f.read().strip().split("\n")]
_sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
_train_indices, _val_indices = next(_sgkf.split(
    list(range(len(_all_labels))), _all_labels, _scene_groups,
))
_train_indices, _val_indices = list(_train_indices), list(_val_indices)
print(f"Scene-aware split: train={len(_train_indices)}, val={len(_val_indices)}")

print(f"Dataset: {LOCAL_DATASET_PATH}")


# Define trainable (same for both new and restored runs)
trainable = tune.with_resources(
    tune.with_parameters(
        train_linet_tune,
        data_root=LOCAL_DATASET_PATH,
        norm_stats=norm_stats,
        seed=SEED,
        pretrained_weights_path=PRETRAINED_WEIGHTS_PATH if LOAD_WEIGHTS else None,
        freeze_backbone_epochs=FREEZE_BACKBONE_EPOCHS if LOAD_WEIGHTS else 0,
        freeze_backbone_lr=FREEZE_BACKBONE_LR,
        train_indices=_train_indices,
        val_indices=_val_indices,
    ),
    resources={"cpu": 3, "gpu": 1.0 / 14},
)


# Callback to force-sync experiment state to Drive
drive_sync_cb = DriveSyncCallback(LOCAL_STORAGE_PATH, DRIVE_STORAGE_PATH, EXPERIMENT_NAME)


if RESUME_EXISTING:
    # =========================================================
    # RESUME: Restore previous experiment from Google Drive
    # =========================================================
    print("\n" + "=" * 60)
    print("RESUMING EXPERIMENT FROM GOOGLE DRIVE")
    print("=" * 60)

    tuner = tune.Tuner.restore(
        path=local_experiment_path,
        trainable=trainable,
        resume_unfinished=True,
        resume_errored=True,
    )

else:
    # =========================================================
    # NEW: Create fresh experiment
    # =========================================================
    print("\n" + "=" * 60)
    print("STARTING NEW EXPERIMENT")
    print("=" * 60)

    # Search space (SUN RGB-D tuned ranges)
    search_space = {
        # Learning rates
        "lr": tune.loguniform(1.0e-4, 3e-4),
        "wd": tune.loguniform(1e-4, 1e-2),

        # Scheduler eta_min
        "eta_min": tune.loguniform(1.0e-6, 1.0e-5),

        # Regularization
        "dropout_p": tune.quniform(0.5, 0.7, 0.01),
        "label_smoothing": tune.quniform(0.05, 0.2, 0.01),
        "grad_clip_norm": tune.quniform(0.8, 1.7, 0.1),

        # Stem learning rate multiplier (DLR for conv1)
        "stem_lr_multiplier": tune.quniform(0.5, 5.0, 0.1),

        # Augmentation parameters
        "rgb_aug_prob": tune.quniform(0.9, 1.4, 0.01),
        "rgb_aug_mag": tune.quniform(1.0, 1.4, 0.01),
        "depth_aug_prob": tune.quniform(0.9, 1.4, 0.01),
        "depth_aug_mag": tune.quniform(1.0, 1.4, 0.01),

        # Modality dropout
        "modality_dropout_rate": tune.quniform(0.45, 0.55, 0.01),
    }



    reporter = CLIReporter(
        parameter_columns=[
            "lr",
            "wd",
            "eta_min",
            # "t_max", "batch_size",
            "dropout_p",
            "label_smoothing",
            "grad_clip_norm",
            'stem_lr_multiplier',
            "rgb_aug_prob",
            "rgb_aug_mag",
            "depth_aug_prob",
            "depth_aug_mag",
            "modality_dropout_rate",
            # "modality_dropout_start",
            #"modality_dropout_ramp",
        ],
        metric_columns={
            "training_iteration": "iter",
            "val_mca": "val_mca",
            "best_val_mca": "best_val_mca",
            "best_train_mca": "best_train_mca",
            "best_accuracy": "best_accuracy",
            "composite": "composite",
            "gap": "gap",
            "best_epoch": "best_epoch",
        },
        max_report_frequency=30,
        print_intermediate_tables=True,
    )

    optuna_search = OptunaSearch(
        metric="composite",
        mode="max",
        sampler=TPESampler(n_startup_trials=15),
    )

    asha_scheduler = ASHAScheduler(
        time_attr="training_iteration",
        metric="composite",
        mode="max",
        max_t=30,
        grace_period=15,
        reduction_factor=2,
    )

    # msr_scheduler = MedianStoppingRule(
    #     time_attr="training_iteration",
    #     metric="best_val_mca",
    #     mode="max",
    #     grace_period=15,
    #     min_samples_required=3,
    # )

    tuner = tune.Tuner(
        trainable,
        param_space=search_space,
        tune_config=tune.TuneConfig(
            scheduler=asha_scheduler,
            search_alg=optuna_search,
            num_samples=NUM_SAMPLES,
            max_concurrent_trials=14,
        ),
        run_config=ray.tune.RunConfig(
            storage_path=LOCAL_STORAGE_PATH,
            name=EXPERIMENT_NAME,
            progress_reporter=reporter,
            verbose=1,
            callbacks=[drive_sync_cb, BestTrialReporter(every_n_results=40)],
        ),
    )


# Run (or resume) tuning
print("\n" + "=" * 60)
print("STARTING HYPERPARAMETER TUNING")
print("=" * 60)

results = tuner.fit()

best_result = results.get_best_result("composite", "max")

print("\n" + "=" * 60)
print("TUNING COMPLETE")
print("=" * 60)
print(f"Best Trial Config: {best_result.config}")
print(f"Best Trial Val MCA: {best_result.metrics['best_val_mca']:.4f}")
print(f"Best Trial Accuracy: {best_result.metrics['best_accuracy']:.4f}")
print(f"Best Trial Loss: {best_result.metrics['best_loss']:.4f}")
print(f"\nExperiment saved to: {local_experiment_path}")
print(f"Drive backup: {experiment_path}")
print(f"To resume after Colab dies: just re-run this notebook.")


2026-04-23 07:01:34,280	INFO worker.py:2012 -- Started a local Ray instance.
[I 2026-04-23 07:01:36,220] A new study created in memory with name: optuna


Scene-aware split: train=3898, val=947
Dataset: /dev/shm/sunrgbd_19_traintest

STARTING NEW EXPERIMENT

STARTING HYPERPARAMETER TUNING
== Status ==
Current time: 2026-04-23 07:01:36 (running for 00:00:00.11)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 30.000: None | Iter 15.000: None
Logical resource usage: 3.0/48 CPUs, 0.07142857142857142/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2026-04-23_07-01-31_547569_4748/artifacts/2026-04-23_07-01-36/scannet_sun_rgbd_hpo_MD/driver_artifacts
Number of trials: 1/300 (1 PENDING)
+---------------------------+----------+-------+-------------+------------+-------------+-------------+-------------------+------------------+----------------------+----------------+---------------+------------------+-----------------+------------------------+
| Trial name                | status   | loc   |          lr |         wd |     eta_min |   dropout_p |   label_smoothing |   grad_clip_norm |   stem_lr_multiplier |   rgb_aug_prob |

2026-04-23 07:02:37,156	WARNING util.py:202 -- The `callbacks.on_trial_result` operation took 11.462 s, which may be a performance bottleneck.
2026-04-23 07:02:37,157	WARNING util.py:202 -- The `process_trial_result` operation took 11.463 s, which may be a performance bottleneck.
2026-04-23 07:02:37,157	WARNING util.py:202 -- Processing trial results took 11.463 s, which may be a performance bottleneck. Please consider reporting results less frequently to Ray Tune.
2026-04-23 07:02:37,157	WARNING util.py:202 -- The `process_trial_result` operation took 11.464 s, which may be a performance bottleneck.


[DriveSyncCallback] synced to Drive (periodic, iter=1)
== Status ==
Current time: 2026-04-23 07:02:37 (running for 00:01:00.94)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 30.000: None | Iter 15.000: None
Logical resource usage: 36.0/48 CPUs, 0.8571428571428569/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2026-04-23_07-01-31_547569_4748/artifacts/2026-04-23_07-01-36/scannet_sun_rgbd_hpo_MD/driver_artifacts
Number of trials: 12/300 (1 PENDING, 11 RUNNING)
+---------------------------+----------+--------------------+-------------+-------------+-------------+-------------+-------------------+------------------+----------------------+----------------+---------------+------------------+-----------------+------------------------+--------+-----------+----------------+------------------+-----------------+-------------+-----------+--------------+
| Trial name                | status   | loc                |          lr |          wd |     eta_min |   dropout_p |  

(train_linet_tune pid=200185) /content/Multi-Stream-Neural-Networks/src/training/schedulers.py:193: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
(train_linet_tune pid=200185)   scheduler.step()


...

== Status ==
Current time: 2026-04-23 11:33:17 (running for 04:31:41.53)
Using AsyncHyperBand: num_stopped=294
Bracket: Iter 30.000: 0.6148509352842423 | Iter 15.000: 0.6027706444651327
Logical resource usage: 6.0/48 CPUs, 0.14285714285714285/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2026-04-23_07-01-31_547569_4748/artifacts/2026-04-23_07-01-36/scannet_sun_rgbd_hpo_MD/driver_artifacts
Number of trials: 300/300 (2 RUNNING, 298 TERMINATED)
+---------------------------+------------+--------------------+-------------+-------------+-------------+-------------+-------------------+------------------+----------------------+----------------+---------------+------------------+-----------------+------------------------+--------+-----------+----------------+------------------+-----------------+-------------+-----------+--------------+
| Trial name                | status     | loc                |          lr |          wd |     eta_min |   dropout_p |   label_smoothing

(train_linet_tune pid=344400) [2026-04-23 11:33:32,555 E 344400 344496] logging.cc:125: Stack trace: 
(train_linet_tune pid=344400)  /usr/local/lib/python3.12/dist-packages/ray/_raylet.so(+0x17a1d8a) [0x7f4ab13add8a] ray::operator<<()
(train_linet_tune pid=344400) /usr/local/lib/python3.12/dist-packages/ray/_raylet.so(+0x17a283c) [0x7f4ab13ae83c] ray::RayLog::operator<< <>()
(train_linet_tune pid=344400) /usr/local/lib/python3.12/dist-packages/ray/_raylet.so(+0x17a4df8) [0x7f4ab13b0df8] ray::TerminateHandler()
(train_linet_tune pid=344400) /lib/x86_64-linux-gnu/libstdc++.so.6(+0xae20c) [0x7f4aafa8920c]
(train_linet_tune pid=344400) /lib/x86_64-linux-gnu/libstdc++.so.6(+0xae277) [0x7f4aafa89277]
(train_linet_tune pid=344400) /lib/x86_64-linux-gnu/libstdc++.so.6(__gxx_personality_v0+0x23c) [0x7f4aafa88afc] __gxx_personality_v0
(train_linet_tune pid=344400) /lib/x86_64-linux-gnu/libgcc_s.so.1(+0x16a06) [0x7f4aaf9d1a06]
(train_linet_tune pid=344400) /lib/x86_64-linux-gnu/libgcc_s.so.1(_Unw

[DriveSyncCallback] synced to Drive (trial complete)
== Status ==
Current time: 2026-04-23 11:33:48 (running for 04:32:12.28)
Using AsyncHyperBand: num_stopped=296
Bracket: Iter 30.000: 0.6147642574062755 | Iter 15.000: 0.6027706444651327
Logical resource usage: 3.0/48 CPUs, 0.07142857142857142/1 GPUs (0.0/1.0 accelerator_type:RTX)
Result logdir: /tmp/ray/session_2026-04-23_07-01-31_547569_4748/artifacts/2026-04-23_07-01-36/scannet_sun_rgbd_hpo_MD/driver_artifacts
Number of trials: 300/300 (300 TERMINATED)
+---------------------------+------------+--------------------+-------------+-------------+-------------+-------------+-------------------+------------------+----------------------+----------------+---------------+------------------+-----------------+------------------------+--------+-----------+----------------+------------------+-----------------+-------------+-----------+--------------+
| Trial name                | status     | loc                |          lr |          wd |    

2026-04-23 11:33:57,623	INFO tune.py:1033 -- Total run time: 16341.41 seconds (16332.29 seconds for the tuning loop).


[DriveSyncCallback] synced to Drive (experiment end)

TUNING COMPLETE
Best Trial Config: {'lr': 0.00023730311223905422, 'wd': 0.0007865114370729337, 'eta_min': 2.903402721942234e-06, 'dropout_p': 0.66, 'label_smoothing': 0.05, 'grad_clip_norm': 1.1, 'stem_lr_multiplier': 3.6, 'rgb_aug_prob': 0.93, 'rgb_aug_mag': 1.2, 'depth_aug_prob': 1.06, 'depth_aug_mag': 1.33, 'modality_dropout_rate': 0.55}
Best Trial Val MCA: 0.6506
Best Trial Accuracy: 0.7086
Best Trial Loss: 1.2514

Experiment saved to: /content/ray_results/scannet_sun_rgbd_hpo_MD
Drive backup: /content/drive/MyDrive/ray_tune_experiments/scannet_sun_rgbd_hpo_MD
To resume after Colab dies: just re-run this notebook.


In [28]:
# =============================================================================
# SAVE RESULTS CSV (for offline analysis)
# =============================================================================
# Ray Tune already saved everything to DRIVE_STORAGE_PATH.
# This cell just exports a clean CSV for easy analysis.
# =============================================================================

import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

results_df = results.get_dataframe()

csv_dir = f"{DRIVE_STORAGE_PATH}/analysis"
Path(csv_dir).mkdir(parents=True, exist_ok=True)

csv_path = f"{csv_dir}/sun_hpo_results_{timestamp}.csv"
results_df.to_csv(csv_path, index=False)
print(f"Results CSV saved: {csv_path}")
print(f"  Trials: {len(results_df)}")

latest_path = f"{csv_dir}/sun_hpo_results_latest.csv"
results_df.to_csv(latest_path, index=False)
print(f"Latest copy: {latest_path}")


Results CSV saved: /content/drive/MyDrive/ray_tune_experiments/analysis/sun_hpo_results_20260423_113357.csv
  Trials: 300
Latest copy: /content/drive/MyDrive/ray_tune_experiments/analysis/sun_hpo_results_latest.csv


In [ ]:
# Analyze Top 10 Trials from Ray Tune (ranked by best_accuracy)
# With continuous search spaces, each trial has unique float values —
# no grouping by config. Treat fold assignment as noise.
import pandas as pd

print("=" * 80)
print("TOP 10 TRIALS BY BEST VAL MCA (CONTINUOUS SEARCH)")
print("=" * 80)

# Get all trials and convert to DataFrame
df = results.get_dataframe()

# Sort by best_accuracy (descending)
df_sorted = df.sort_values('composite', ascending=False)

# Select relevant columns for display
display_cols = [
    'val_mca', 'best_val_mca', 'best_train_mca', 'best_accuracy', 'best_train_acc', 'composite', 'gap', 'best_epoch',
    'config/lr',
    'config/wd',
    'config/eta_min',
    # 'config/t_max',
    'config/dropout_p',
    'config/label_smoothing',
    'config/grad_clip_norm',
    'config/rgb_aug_prob',
    'config/rgb_aug_mag',
    'config/depth_aug_prob',
    'config/depth_aug_mag',
    'config/stem_lr_multiplier',
    'config/modality_dropout_rate',
    # 'config/modality_dropout_start', 'config/modality_dropout_ramp',
]

# Get top 10 trials
top_10 = df_sorted[display_cols].head(100)

# Format for better display
top_10_formatted = top_10.copy()
top_10_formatted['val_mca'] = top_10_formatted['val_mca'].apply(lambda x: f"{x*100:.2f}%")
top_10_formatted['best_val_mca'] = top_10_formatted['best_val_mca'].apply(lambda x: f"{x*100:.2f}%")
top_10_formatted['best_train_mca'] = top_10_formatted['best_train_mca'].apply(lambda x: f"{x*100:.2f}%")
top_10_formatted['best_accuracy'] = top_10_formatted['best_accuracy'].apply(lambda x: f"{x*100:.2f}%")

# Format scientific notation columns
sci_cols = [
    'config/lr', 'config/wd', 'config/eta_min',
]
for col in sci_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.3e}")

# Format float columns
float_cols = [
    'config/dropout_p',
    'config/label_smoothing',
    'config/grad_clip_norm',
    'config/rgb_aug_prob',
    'config/rgb_aug_mag',
    'config/depth_aug_prob',
    'config/depth_aug_mag',
    'config/stem_lr_multiplier',
    'config/modality_dropout_rate',
]
for col in float_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.2f}")

# REMOVE 'config/' PREFIX FROM ALL COLUMN NAMES
top_10_formatted.columns = top_10_formatted.columns.str.replace('config/', '', regex=False)

print(top_10_formatted.to_string(index=False))
print("\n" + "=" * 80)


TOP 10 TRIALS BY BEST VAL MCA (CONTINUOUS SEARCH)
val_mca best_val_mca best_train_mca best_accuracy  best_train_acc  composite      gap  best_epoch config/lr config/wd config/eta_min config/dropout_p config/label_smoothing config/grad_clip_norm config/rgb_aug_prob config/rgb_aug_mag config/depth_aug_prob config/depth_aug_mag config/stem_lr_multiplier config/modality_dropout_rate
 63.54%       65.06%         80.06%        70.86%        0.821960   0.643899 0.149965          18 2.373e-04 7.865e-04      2.903e-06             0.66                   0.05                  1.10                0.93               1.20                  1.06                 1.33                      3.60                         0.55
 65.04%       65.39%         82.85%        70.64%        0.826321   0.643204 0.174667          25 2.048e-04 1.255e-03      2.414e-06             0.69                   0.06                  0.90                1.04               1.24                  1.00                 1.29          

In [30]:
# =============================================================================
# ANALYZE TOP 10 TRIALS BY BEST VAL MCA
# =============================================================================

import pandas as pd

df = results.get_dataframe()
df = df.sort_values("gap", ascending=True)
top_10 = df.head(10).copy()

config_cols = [c for c in df.columns if c.startswith("config/")]

print("=" * 80)
print("TOP 10 TRIALS BY BEST VAL MCA")
print("=" * 80)

for rank, (_, row) in enumerate(top_10.iterrows(), 1):
    gap = row.get("best_train_mca", 0) - row.get("best_val_mca", 0)
    print(f"\n--- #{rank} | Best Val MCA: {row['best_val_mca']*100:.2f}% | "
          f"Val MCA: {row['best_val_mca']*100:.2f}% | Acc: {row['best_accuracy']*100:.2f}% | "
          f"Gap: {gap*100:.1f}pp ---")

print("\n" + "=" * 80)
print("HYPERPARAMETER RANGES ACROSS TOP 10")
print("=" * 80)
print(f"{'Parameter':<35} {'Min':>12} {'Max':>12} {'Median':>12}")
print("-" * 75)

for col in config_cols:
    short_name = col.replace("config/", "")
    col_min = top_10[col].min()
    col_max = top_10[col].max()
    col_med = top_10[col].median()
    if abs(col_med) < 0.001:
        print(f"{short_name:<35} {col_min:>12.2e} {col_max:>12.2e} {col_med:>12.2e}")
    else:
        print(f"{short_name:<35} {col_min:>12.4f} {col_max:>12.4f} {col_med:>12.4f}")


TOP 10 TRIALS BY BEST VAL MCA

--- #1 | Best Val MCA: 59.84% | Val MCA: 59.84% | Acc: 63.89% | Gap: -0.4pp ---

--- #2 | Best Val MCA: 58.33% | Val MCA: 58.33% | Acc: 63.46% | Gap: 0.9pp ---

--- #3 | Best Val MCA: 58.70% | Val MCA: 58.70% | Acc: 63.89% | Gap: 0.9pp ---

--- #4 | Best Val MCA: 60.17% | Val MCA: 60.17% | Acc: 64.10% | Gap: 1.1pp ---

--- #5 | Best Val MCA: 59.91% | Val MCA: 59.91% | Acc: 63.57% | Gap: 1.5pp ---

--- #6 | Best Val MCA: 59.98% | Val MCA: 59.98% | Acc: 65.47% | Gap: 1.7pp ---

--- #7 | Best Val MCA: 58.82% | Val MCA: 58.82% | Acc: 62.72% | Gap: 2.3pp ---

--- #8 | Best Val MCA: 59.00% | Val MCA: 59.00% | Acc: 64.84% | Gap: 3.3pp ---

--- #9 | Best Val MCA: 59.66% | Val MCA: 59.66% | Acc: 63.99% | Gap: 3.4pp ---

--- #10 | Best Val MCA: 59.44% | Val MCA: 59.44% | Acc: 63.89% | Gap: 3.5pp ---

HYPERPARAMETER RANGES ACROSS TOP 10
Parameter                                    Min          Max       Median
--------------------------------------------------------